# theta_interpretation: entropy convergence vs. $\sigma$

Goal: sweep the SDE diffusion coefficient `sigma` for the **correctly-specified**
scalar Gaussian case (`terms=['x2']`) and reproduce a three-curve plot vs.
$\sigma^2$ (log x-axis):

- $H(p_*)$ -- the *exact* entropy of the true max-entropy distribution. We
  can compute this in closed form here (not just bound it) precisely
  *because* the potential set is correctly specified: for `phi(x) = x^2`
  and target `N(0, data_sigma^2)`, the max-entropy distribution matching
  `E[x^2] = data_sigma^2` **is** `N(0, data_sigma^2)` itself, whose entropy
  is `0.5 * log(2*pi*e*data_sigma^2)`. Constant in `sigma` (the SDE
  diffusion coefficient) -- it depends only on `data_sigma`.
- $H(p_1^\sigma)$ -- the value of H obtained by using the the p built from theta_1
  found by MGD (and MGD regularized) and the potentials chosen for the run, and
  the shape of the maximum entropy distribution
- H^sigma_star -- entropy bound computed by MGD 


In [11]:
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.integrate import quad
from IPython.display import Image, display

root = Path.cwd()
if not (root / 'run_SDE.py').exists():
    root = root / 'theta_interpretation'  # allow running the notebook from the repo root
sys.path.insert(0, str(root))
sys.path.insert(0, str(root.parent / 'codes'))

from run_SDE import make_args, run_and_diagnose, get_scalar_potentials, device
from utils_entropy import entropy_bound, standard_gaussian_entropy  # codes/utils_entropy.py

print('device:', device)

device: cuda


## Parameters

`SIGMA_LIST` is the swept quantity (SDE diffusion coefficient); `SEED_LIST`
(`N_SEEDS` seeds) is the second swept axis, so every downstream quantity
can be reported as a mean +/- std across seeds per sigma instead of a
single noisy point. Everything else is a fixed override threaded through
`make_args(['x2'], sigma=..., seed=..., **PARAM_OVERRIDES)` -- see
`theta_interpretation.ipynb`'s parameter-override cell for why this is
necessary (`make_args()`'s un-overridden defaults mirror the CLI defaults,
not anything you'd type in this cell).

In [12]:
SIGMA_LIST = np.geomspace(0.3, 6.0, 10)   # sigma^2 spans roughly [0.1, 36], log-spaced
print("sigma^2 list:", SIGMA_LIST)

N_SEEDS = 100                              # seeds per sigma -- adjust as needed
SEED_LIST = list(range(N_SEEDS))

N1 = 50000
DATA_SIGMA = 1.0
NT = 1000
SCHEDULE_EXPONENT = 2
INTERPOLANT = 'Cos'
REGULARIZATION = 1e-3
LAM = 1e-08
N_SUBSAMPLE = 1
BATCH_SIZE = None
N_BINS = 30                 # histogram bins for both run_SDE.py's own diagnostics AND the H^1 estimator below
MOMENT_THRESHOLD = 1e-8
FORCE_RERUN = False
NO_SAVE_AUX_MOMENTS = False

# seed is swept via SEED_LIST below, not fixed here.
PARAM_OVERRIDES = dict(
    n1=N1, data_sigma=DATA_SIGMA, nt=NT,
    schedule_exponent=SCHEDULE_EXPONENT, interpolant=INTERPOLANT,
    regularization=REGULARIZATION, lam=LAM, n_subsample=N_SUBSAMPLE,
    batch_size=BATCH_SIZE, n_bins=N_BINS, moment_threshold=MOMENT_THRESHOLD,
    force_rerun=FORCE_RERUN, no_save_aux_moments=NO_SAVE_AUX_MOMENTS,
    label='entropy_convergence',
)


sigma^2 list: [0.3        0.41848524 0.58376632 0.81432528 1.1359437  1.58458557
 2.2104189  3.0834256  4.30122699 6.        ]


## Sweep `sigma` x `seed`, run the experiment for each pair

Each `(sigma, seed)` pair gets its own config folder (`build_config_name()`
hashes both `sigma` and `seed` into the name), so re-running this cell
after the first time just reloads the cached results instead of
re-fitting.

In [ ]:
runs = {}
for sigma_val in SIGMA_LIST:
    runs[float(sigma_val)] = {}
    for seed in SEED_LIST:
        args = make_args(['x2'], sigma=float(sigma_val), seed=seed, outdir=str(root), **PARAM_OVERRIDES)
        runs[float(sigma_val)][seed] = run_and_diagnose(args)
    print(f"sigma={sigma_val:.4f}: {len(SEED_LIST)} seeds done")


2026-09-04 12:51:45  INFO      Reusing existing experiment: thetainterp_sigmadata1.0_sigma0.3_nt1000_n1_50000_lam1e-08_seed_0_terms8e683187_entropy_convergence (no new folder created)
2026-09-04 12:51:46  INFO      Final theta (order matches ['x2']): [0.0]
2026-09-04 12:51:46  INFO      Reference (unverified against this run): for target N(0, data_sigma^2) under p(x) ~ exp(sum_i theta_i * phi_i(x)), theta on x^2 is expected near -0.500000. Fitted theta_x2[-1] = 0.000000
2026-09-04 12:51:46  INFO      Saved diagnostic figures to /lustre/fswork/projects/rech/wbg/ukv59en/MGD-for-Maximum-Entropy-Generation/theta_interpretation/experiments/thetainterp_sigmadata1.0_sigma0.3_nt1000_n1_50000_lam1e-08_terms8e683187_entropy_convergence/thetainterp_sigmadata1.0_sigma0.3_nt1000_n1_50000_lam1e-08_seed_0_terms8e683187_entropy_convergence/figures
2026-09-04 12:51:46  INFO      Config: thetainterp_sigmadata1.0_sigma0.3_nt1000_n1_50000_lam1e-08_seed_1_terms8e683187_entropy_convergence
2026-09-04 12:51:

1000it [00:02, 426.46it/s]


Loop finished
After loop: CPU=1.00 GB | GPU alloc=0.04 GB | GPU reserved=0.06 GB
Preparing regularised solve
Before _solve_regularised: CPU=1.00 GB | GPU alloc=0.04 GB | GPU reserved=0.06 GB
Dropped close-in-time nodes: 3
Last times: [0.99993598 0.999951   0.999964   0.99997503 0.999991  ]
Last dt: [1.50203705e-05 1.29938126e-05 1.10268593e-05 1.59740448e-05]
Calling torch.linalg.solve
Before solve: CPU=1.00 GB | GPU alloc=0.05 GB | GPU reserved=0.06 GB
Solve finished
After solve: CPU=1.00 GB | GPU alloc=0.05 GB | GPU reserved=0.06 GB
After _solve_regularised: CPU=1.00 GB | GPU alloc=0.04 GB | GPU reserved=0.06 GB
Stacking outputs
Everything stacked: CPU=1.00 GB | GPU alloc=0.04 GB | GPU reserved=0.06 GB
Returning
2026-09-04 12:51:49  INFO      SDE integration finished in 3.1 s
2026-09-04 12:51:50  INFO      Final theta (order matches ['x2']): [9.784953117370605]
2026-09-04 12:51:50  INFO      Reference (unverified against this run): for target N(0, data_sigma^2) under p(x) ~ exp(sum_i

1000it [00:02, 430.86it/s]


Loop finished
After loop: CPU=1.02 GB | GPU alloc=0.04 GB | GPU reserved=0.06 GB
Preparing regularised solve
Before _solve_regularised: CPU=1.02 GB | GPU alloc=0.04 GB | GPU reserved=0.06 GB
Dropped close-in-time nodes: 3
Last times: [0.99993598 0.999951   0.999964   0.99997503 0.999991  ]
Last dt: [1.50203705e-05 1.29938126e-05 1.10268593e-05 1.59740448e-05]
Calling torch.linalg.solve
Before solve: CPU=1.02 GB | GPU alloc=0.05 GB | GPU reserved=0.06 GB
Solve finished
After solve: CPU=1.02 GB | GPU alloc=0.05 GB | GPU reserved=0.06 GB
After _solve_regularised: CPU=1.02 GB | GPU alloc=0.04 GB | GPU reserved=0.06 GB
Stacking outputs
Everything stacked: CPU=1.02 GB | GPU alloc=0.04 GB | GPU reserved=0.06 GB
Returning
2026-09-04 12:51:53  INFO      SDE integration finished in 3.1 s
2026-09-04 12:51:54  INFO      Final theta (order matches ['x2']): [-9.056035041809082]
2026-09-04 12:51:54  INFO      Reference (unverified against this run): for target N(0, data_sigma^2) under p(x) ~ exp(sum_

...

Loop finished
After loop: CPU=18.92 GB | GPU alloc=0.32 GB | GPU reserved=0.34 GB
Preparing regularised solve
Before _solve_regularised: CPU=18.92 GB | GPU alloc=0.32 GB | GPU reserved=0.34 GB
Dropped close-in-time nodes: 3
Last times: [0.99993598 0.999951   0.999964   0.99997503 0.999991  ]
Last dt: [1.50203705e-05 1.29938126e-05 1.10268593e-05 1.59740448e-05]
Calling torch.linalg.solve
Before solve: CPU=18.92 GB | GPU alloc=0.33 GB | GPU reserved=0.34 GB
Solve finished
After solve: CPU=18.92 GB | GPU alloc=0.33 GB | GPU reserved=0.34 GB
After _solve_regularised: CPU=18.92 GB | GPU alloc=0.32 GB | GPU reserved=0.34 GB
Stacking outputs
Everything stacked: CPU=18.92 GB | GPU alloc=0.32 GB | GPU reserved=0.34 GB
Returning
2026-09-04 17:43:42  INFO      SDE integration finished in 44.8 s
2026-09-04 17:43:43  INFO      Final theta (order matches ['x2']): [-1.230366587638855]
2026-09-04 17:43:43  INFO      Reference (unverified against this run): for target N(0, data_sigma^2) under p(x) ~ e

1000it [00:02, 463.96it/s]


Loop finished
After loop: CPU=18.94 GB | GPU alloc=0.32 GB | GPU reserved=0.34 GB
Preparing regularised solve
Before _solve_regularised: CPU=18.94 GB | GPU alloc=0.32 GB | GPU reserved=0.34 GB
Dropped close-in-time nodes: 3
Last times: [0.99993598 0.999951   0.999964   0.99997503 0.999991  ]
Last dt: [1.50203705e-05 1.29938126e-05 1.10268593e-05 1.59740448e-05]
Calling torch.linalg.solve
Before solve: CPU=18.94 GB | GPU alloc=0.33 GB | GPU reserved=0.34 GB
Solve finished
After solve: CPU=18.94 GB | GPU alloc=0.33 GB | GPU reserved=0.34 GB
After _solve_regularised: CPU=18.94 GB | GPU alloc=0.32 GB | GPU reserved=0.34 GB
Stacking outputs
Everything stacked: CPU=18.94 GB | GPU alloc=0.32 GB | GPU reserved=0.34 GB
Returning
2026-09-04 17:44:28  INFO      SDE integration finished in 44.9 s
2026-09-04 17:44:29  INFO      Final theta (order matches ['x2']): [0.2091619074344635]
2026-09-04 17:44:29  INFO      Reference (unverified against this run): for target N(0, data_sigma^2) under p(x) ~ e

1000it [00:02, 437.65it/s]


Loop finished
After loop: CPU=18.97 GB | GPU alloc=0.32 GB | GPU reserved=0.34 GB
Preparing regularised solve
Before _solve_regularised: CPU=18.97 GB | GPU alloc=0.32 GB | GPU reserved=0.34 GB
Dropped close-in-time nodes: 3
Last times: [0.99993598 0.999951   0.999964   0.99997503 0.999991  ]
Last dt: [1.50203705e-05 1.29938126e-05 1.10268593e-05 1.59740448e-05]
Calling torch.linalg.solve
Before solve: CPU=18.96 GB | GPU alloc=0.33 GB | GPU reserved=0.34 GB


## check moment matching

Spot check only, `SEED_LIST[0]` per sigma -- not a per-seed diagnostic
loop (that would be `len(SIGMA_LIST) * N_SEEDS` images).

In [ ]:
for sigma_val in SIGMA_LIST:
    run = runs[float(sigma_val)][SEED_LIST[0]]
    print(f"--- sigma = {sigma_val:.4f}  (sigma^2 = {sigma_val**2:.4f}), seed={SEED_LIST[0]} (spot check) ---")
    display(Image(filename=str(run['fig_dir'] / 'moment_matching.png')))


## Compute the three entropy quantities

- `H_star`: exact, computed once via `standard_gaussian_entropy(1, log_det_cov=2*log(data_sigma))`
  -- `standard_gaussian_entropy(d, log_det_cov)` is `codes/utils_entropy.py`'s
  plain closed-form Gaussian entropy; `log_det_cov = log(data_sigma**2)` for
  our scalar `N(0, data_sigma^2)` target. Constant across sigma AND seed.
- `H_star_bound`: `entropy_bound(...)['H_bound']` per (sigma, seed) -- the
  MGD Prop. 4.3 lower bound (`check_p0=False` here purely to skip that
  extra diagnostic and keep the sweep fast; it doesn't affect `H_bound`
  itself). `entropy_bound`'s own `quadrature_gap` diagnostic (trapezoid -
  left-Riemann on its internal SDE-time integral) is also kept, for the
  bound investigation below.
- `H_p1_raw` / `H_p1_reg` (two variants of $H(p_1^\sigma)$): the *actual*
  differential entropy of the max-entropy density built from the fitted
  Lagrange multipliers, `p_theta1(x) = exp(theta_1^T phi(x)) / Z_theta1`,
  using the run's own potentials `phi`. This is neither a bound nor a
  sample-based histogram estimate -- `theta_1` and `phi` pin down the
  density's analytic functional form exactly (in this scalar,
  correctly-specified case), so `Z_theta1` and
  `H(p_theta1) = -int p_theta1(x) log p_theta1(x) dx` are evaluated
  directly via 1D numerical quadrature. Two variants because `theta_1`
  itself has two candidates per run: `theta_t[-1]` (raw, per-step MGD fit)
  and `Theta_reg[-1]` (the time-regularised fit).

`entropy_from_theta_quad`/`_is_normalizable` below now reuse the SAME exact
normalizability test and mode-aware, support-truncated quadrature as
`theta_interpretation_analysis.ipynb`'s `log_Z_quad` -- the previous
version of this cell had its own, separately-maintained, cruder
"coarse-grid argmax at the boundary" check and integrated over the full
fixed `x_range` unconditionally, which could silently drift from the
analysis notebook's (better) approach.

Every quantity is computed per `(sigma, seed)`, then reduced to a
per-sigma mean +/- std over the `np.isfinite` seeds (`_mean_std` below,
same one-mask-for-count-and-aggregate discipline as the analysis
notebook's fixed §3.5/§4.4) -- `H_p1_raw` in particular is expected to
have some seeds go non-normalizable (see the bound-investigation section
below for why that's not surprising).

In [ ]:
def _degree(term):
    return int(term[1:])  # 'x2' -> 2, 'x4' -> 4


def log_p_unnorm_theta(x, theta, potentials):
    """theta^T phi(x), phi built from the run's own potentials.forward() --
    matches the term order of potentials.values() (== theta's own coefficient
    order, since theta is fit against exactly this potentials dict)."""
    x_t = torch.as_tensor(np.asarray(x), dtype=torch.float64)
    val = torch.zeros_like(x_t)
    for coeff, pot in zip(theta, potentials.values()):
        val = val + float(coeff) * pot.forward(x_t)
    return val.numpy()


def _is_normalizable(theta, term_names):
    """Exact analytic test, ported unchanged from
    theta_interpretation_analysis.ipynb: exp(theta^T phi) is integrable iff
    the highest-degree term with a nonzero coefficient has even degree and
    a negative coefficient. Replaces this cell's previous, separately
    maintained 'coarse-grid argmax at the boundary' heuristic."""
    live = [(_degree(t), theta[i]) for i, t in enumerate(term_names) if theta[i] != 0.0]
    if not live:
        return False
    deg, coef = max(live)
    return deg % 2 == 0 and coef < 0


def entropy_from_theta_quad(theta, term_names, potentials, x_range=(-50.0, 50.0), label=''):
    """Exact differential entropy of p_theta(x) = exp(theta^T phi(x)) / Z_theta
    via 1D quadrature -- the actual max-entropy density shaped by the fitted
    theta, not a bound and not a sample-histogram estimate. Support
    truncated to where the log-density doesn't underflow and `points=` the
    density's local modes, same as theta_interpretation_analysis.ipynb's
    log_Z_quad, for better quadrature accuracy at peaks."""
    theta = np.asarray(theta, dtype=np.float64)
    if not _is_normalizable(theta, term_names):
        print(f"  WARNING entropy_from_theta_quad{' ' + label if label else ''}: "
              f"non-normalizable theta={theta}. Returning nan.")
        return float('nan')

    grid = np.linspace(*x_range, 4001)
    log_vals = log_p_unnorm_theta(grid, theta, potentials)
    m = log_vals.max()
    support = grid[log_vals - m > -700.0]          # everything else underflows
    lo, hi = support[0], support[-1]
    modes = list(grid[1:-1][(log_vals[1:-1] > log_vals[:-2]) &
                            (log_vals[1:-1] > log_vals[2:])])

    Z_rel, _ = quad(lambda x: np.exp(log_p_unnorm_theta(x, theta, potentials) - m),
                     lo, hi, points=modes, limit=200)
    if Z_rel <= 0 or not np.isfinite(Z_rel):
        print(f"  WARNING entropy_from_theta_quad{' ' + label if label else ''}: quad "
              f"integral non-positive/non-finite (val={Z_rel}) for theta={theta}. Returning nan.")
        return float('nan')
    log_Z = m + np.log(Z_rel)

    def neg_p_logp(x):
        lp = log_p_unnorm_theta(x, theta, potentials) - log_Z
        return -np.exp(lp) * lp

    H_val, _ = quad(neg_p_logp, lo, hi, points=modes, limit=200)
    return H_val


def _mean_std(mat):
    """Per-row (per-sigma) mean/std/n_valid over the isfinite seeds in that
    row -- one np.isfinite mask reused for the count and the aggregate."""
    n_rows = mat.shape[0]
    mean = np.full(n_rows, np.nan)
    std = np.full(n_rows, np.nan)
    n_valid = np.zeros(n_rows, dtype=int)
    for i in range(n_rows):
        valid = np.isfinite(mat[i])
        n_valid[i] = int(valid.sum())
        if n_valid[i]:
            mean[i] = mat[i, valid].mean()
        if n_valid[i] > 1:
            std[i] = mat[i, valid].std(ddof=1)
    return mean, std, n_valid


H_star = float(standard_gaussian_entropy(1, log_det_cov=2 * np.log(DATA_SIGMA)))
print('H(p_*) =', H_star)

sigmas_sorted = np.array(sorted(runs.keys()))

H_star_bound_mat = np.full((len(sigmas_sorted), N_SEEDS), np.nan)
H_p1_raw_mat = np.full((len(sigmas_sorted), N_SEEDS), np.nan)
H_p1_reg_mat = np.full((len(sigmas_sorted), N_SEEDS), np.nan)
quad_gap_mat = np.full((len(sigmas_sorted), N_SEEDS), np.nan)   # entropy_bound's own trapz-vs-left_riemann diagnostic

for i, sigma_val in enumerate(sigmas_sorted):
    for j, seed in enumerate(SEED_LIST):
        run = runs[sigma_val][seed]
        potentials = get_scalar_potentials(run['args'].terms)
        term_names = list(potentials.keys())

        eb = entropy_bound({'run': run['result']}, 'run', potentials, device=device, check_p0=False)
        H_star_bound_mat[i, j] = eb['H_bound']
        quad_gap_mat[i, j] = eb['quadrature_gap']

        theta_raw = run['result']['theta_t'][-1].detach().cpu().numpy()
        theta_reg = run['result']['Theta_reg'][-1].detach().cpu().numpy()
        H_p1_raw_mat[i, j] = entropy_from_theta_quad(theta_raw, term_names, potentials,
                                                       label=f'raw sigma={sigma_val:.4f} seed={seed}')
        H_p1_reg_mat[i, j] = entropy_from_theta_quad(theta_reg, term_names, potentials,
                                                       label=f'reg sigma={sigma_val:.4f} seed={seed}')

    print(f"sigma={sigma_val:.4f}  "
          f"H_star_bound mean={np.nanmean(H_star_bound_mat[i]):.4f}  "
          f"H_p1_raw (n_valid={int(np.isfinite(H_p1_raw_mat[i]).sum())}/{N_SEEDS}) "
          f"mean={np.nanmean(H_p1_raw_mat[i]):.4f}  "
          f"H_p1_reg mean={np.nanmean(H_p1_reg_mat[i]):.4f}")

H_star_bound_mean, H_star_bound_std, H_star_bound_nvalid = _mean_std(H_star_bound_mat)
H_p1_raw_mean, H_p1_raw_std, H_p1_raw_nvalid = _mean_std(H_p1_raw_mat)
H_p1_reg_mean, H_p1_reg_std, H_p1_reg_nvalid = _mean_std(H_p1_reg_mat)


## Investigating: why is `H_star_bound` above `H(p_*)`?

`entropy_bound`'s own docstring (`codes/utils_entropy.py`) says exactly
what it bounds: "Lower bound on `H(p_1^sigma)`" -- the entropy of the
ACTUAL SDE trajectory's endpoint distribution at `i_final` (default the
last `theta_t` step). NOT `H(p_*)` (the true, data-generating maxent
entropy), and NOT whatever `entropy_from_theta_quad` computes from
whichever `theta_1` you hand it. All three only coincide once the fit has
fully converged -- so "the bound exceeds X" is only a real problem for
whichever X the bound actually targets.

Two different gaps are being conflated when this looks alarming; they
have different explanations:

- **`H_star_bound - H_star`**: this is the one that's actually about the
  bound's own validity, and it should be small and (up to numerical
  slack) non-negative -- `H(p_1^sigma)` only equals `H(p_*)` exactly if
  the SDE has fully converged to the true maxent distribution, and
  `entropy_bound`'s own internal SDE-time integral (trapezoid vs.
  left-Riemann) has finite-step discretization slack regardless, exposed
  here as `quadrature_gap`.
- **`H_star_bound` vs. `H_p1_raw`**: NOT a fair comparison. `H_p1_raw` is
  built from `theta_t[-1]`, a single raw last SDE step whose convergence
  is explicitly not verified anywhere in this codebase (see
  `theta_interpretation_analysis.ipynb`'s 3.1) -- it can be, and above
  often is, a poor proxy for `p_1^sigma` (it goes non-normalizable on
  several seeds, and strongly negative-entropy/overconfining on others).
  `H_p1_reg` (`Theta_reg[-1]`, the intended converged-fit proxy) is the
  closer apples-to-apples comparison, though even it isn't exactly the
  bound's target `p_1^sigma`.

In [ ]:
print("H_star_bound - H_star, and entropy_bound's own quadrature_gap (mean over seeds, per sigma):")
for i, sigma_val in enumerate(sigmas_sorted):
    print(f"  sigma={sigma_val:.4f}: bound-H_star={H_star_bound_mean[i] - H_star:+.5f}   "
          f"quadrature_gap (trapz - left_riemann, mean)={np.nanmean(quad_gap_mat[i]):+.5f}")

print()
print("H_star_bound vs the two theta1 proxies (mean over seeds, per sigma) -- "
      "'bound - H_p1_reg' is the fairer validity check (see markdown above):")
for i, sigma_val in enumerate(sigmas_sorted):
    print(f"  sigma={sigma_val:.4f}: "
          f"bound-H_p1_raw={H_star_bound_mean[i] - H_p1_raw_mean[i]:+.4f} "
          f"(n_valid={H_p1_raw_nvalid[i]}/{N_SEEDS})   "
          f"bound-H_p1_reg={H_star_bound_mean[i] - H_p1_reg_mean[i]:+.4f} "
          f"(n_valid={H_p1_reg_nvalid[i]}/{N_SEEDS}, should be <=0 if Theta_reg[-1] ~ p_1^sigma)")


## Reproduce the figure


In [7]:
%matplotlib inline

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

ax.axhline(H_star, color='red', lw=1.5, label=r'$H(p_*)$')

ax.plot(sigmas_sorted**2, H_star_bound_mean, 'o-', color='tab:blue', ms=5,
        label=r'$H_*^\sigma$ (MGD bound)')
ax.fill_between(sigmas_sorted**2, H_star_bound_mean - H_star_bound_std,
                 H_star_bound_mean + H_star_bound_std, color='tab:blue', alpha=0.2)

ax.plot(sigmas_sorted**2, H_p1_raw_mean, 'o--', color='black', ms=5,
        label=r'$H(p_1^\sigma)$ (raw $\theta_1$)')
ax.fill_between(sigmas_sorted**2, H_p1_raw_mean - H_p1_raw_std,
                 H_p1_raw_mean + H_p1_raw_std, color='black', alpha=0.15)

ax.plot(sigmas_sorted**2, H_p1_reg_mean, 's--', color='tab:green', ms=5,
        label=r'$H(p_1^\sigma)$ (reg. $\theta_1$)')
ax.fill_between(sigmas_sorted**2, H_p1_reg_mean - H_p1_reg_std,
                 H_p1_reg_mean + H_p1_reg_std, color='tab:green', alpha=0.15)

ax.set_xscale('log')
ax.set_xlabel(r'$\sigma^2$')
ax.set_ylabel('entropy')
ax.legend(frameon=False)
fig.tight_layout()

out_dir = root / 'entropy_convergence'
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / 'entropy_convergence.png', dpi=150, bbox_inches='tight')
plt.show()